# Lesson 3: Build a complete ACT training batch

**Goal:** turn demonstration episodes into camera inputs, normalized state, and masked action targets that a real ACT model can use. The routine file reader, row ordering, and frozen split lists are supplied. You implement two useful operations: fit training-only statistics, then build a complete batch.

Read the explanations before the exercises. Start from a fresh kernel. [Previous: ACT execution](../02-act-execution/02-act-execution.ipynb) · [Course index](../index.ipynb) · [Next: ACT training](../04-act-training/04-act-training.ipynb)


In [ ]:
import os
import sys
from pathlib import Path

WORKSPACE = next(
    path
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "manifests/act_references.yaml").is_file()
)
sys.path.insert(0, str(WORKSPACE / "src"))
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / ".local-data/hf/datasets")

In [ ]:
from collections.abc import Mapping, Sequence
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from IPython.display import display
from ipywidgets import FloatSlider, interactive
from lerobot.policies.act import ACTPolicy

from vla_lab.policy.act import make_act_config
from vla_lab.teaching.adapters import camera_hwc_to_chw
from vla_lab.teaching.data import read_local_episode_rows
from vla_lab.teaching.live import make_act_raw_model
from vla_lab.teaching.records import CAMERA_KEYS, CameraFrame, FeatureKey, TrainingEpisode


@dataclass(frozen=True)
class MeanStd:
    """Per-feature statistics; ``mean`` and ``std`` both have shape ``(D,)``."""

    mean: np.ndarray
    std: np.ndarray


@dataclass(frozen=True)
class FeatureStats:
    """Training-split statistics for the state (``D=8``) and action (``D=7``) features."""

    states: MeanStd
    actions: MeanStd

## An episode is the unit of separation

A demonstration contains observations and the actions paired with them. A training example starts at frame $s$. ACT receives the cameras and state at $s$, and learns to predict the next $T$ action rows. An episode ends when that demonstration ends. A window must never take a target from the next episode, even if both episodes are adjacent in a file.

```text
frozen episode IDs ── training episodes ── fit mean and standard deviation
                 └── validation episodes ───────────────┐
                                                       │ use training statistics
one episode:   frame s       s+1       s+2        end     │
               cameras,state                           │
               action ───── action ── action ── padding │
                   └── normalize, mask, stack ──────────┴─ ACT batch
```

A random split of frames can put nearly identical neighboring images on both sides. The validation result then measures a much easier problem. The supplied manifest instead keeps each complete episode in one split. Simulator state IDs are a separate split: a demonstration ID is not a simulator initial-state ID.

The source dataset reports a **10 FPS storage label**. LIBERO's source generator uses **20 Hz control**. Use frame offsets $s, s+1, \dots$ to select adjacent rows. Do not infer a physical 0.1-second action duration from the storage label. At runtime, one relative command advances one 0.05-second controller tick. New recordings keep every aligned transition.

## How one episode is stored: `TrainingEpisode`

The supplied reader returns `dict[int, TrainingEpisode]`, which maps an episode ID to that episode. Both record types come from `vla_lab.teaching.records`, so your editor can list their fields:

```python
@dataclass(frozen=True)
class TrainingEpisode:
    actions: np.ndarray               # (L, 7) float32, one row per frame
    states: np.ndarray                # (L, 8) float32, one row per frame
    images: dict[int, CameraFrame]    # frame index -> both camera images at that frame
    frame_count: int                  # property: L

@dataclass(frozen=True)
class CameraFrame:
    image: np.ndarray    # agent-view camera, (3, H, W) float32 in [0, 1]
    image2: np.ndarray   # wrist camera,      (3, H, W) float32 in [0, 1]
```

Where:
* $L$ - number of frames in the episode; row $i$ of `actions` and `states` belongs to frame $i$.
* $H$, $W$ - image height and width in pixels.

**Why images are a separate, sparse mapping.** The numbers and the pixels have different costs. `actions` and `states` are small and cover **every** frame: a window needs the true episode end, and the statistics need all frames. Images come from compressed video, and decoding every frame is slow. The reader therefore decodes images at only a few frames per episode (`image_samples_per_episode`), for example `images.keys() == {0, 55, 111}`. A training example can start only at a frame that has decoded images, so `episode.images[s]` is how you look up the cameras for start $s$. If `TrainingEpisode` had `image` and `image2` arrays covering every frame, it would cost the full decoding time and memory.

## Pixels, state, and action have different conversions

Raw simulator cameras are `uint8` HWC arrays: height, width, RGB channels. The supplied camera converter applies the required orientation and returns CHW `float32` in $[0,1]$. The dataset reader already returns this CHW layout and scale; do not rotate or divide its images by 255 again. This ACT recipe uses identity image normalization after scaling. Diffusion and SmolVLA have different image processing, taught in their sections.

The eight state values are end-effector position (three, in metres), axis-angle orientation (three, in radians), and two gripper finger positions (metres). The dataset reader supplies them in the saved order. The seven action values are bounded relative controller commands: three translation channels, three rotation channels, and one gripper channel. Their controller scaling is not the same as the physical state units. Keep the checkpoint's feature order.

```python
# Raw pixel [0, 128, 255] becomes approximately [0.0, 0.502, 1.0].
```

In [ ]:
pixel = np.array([[[0, 128, 255]]], dtype=np.uint8)
scaled = camera_hwc_to_chw(pixel, rotate_180=False)
print("HWC", pixel.shape, "to CHW", scaled.shape, scaled[:, 0, 0])

## Statistics belong to the training split

For each feature $d$, compute the population mean and standard deviation over **real frames from training episodes**, before window creation:

$$\mu_d=\frac{1}{N}\sum_{i=1}^{N}x_{i,d}$$

$$\sigma_d=\sqrt{\frac{1}{N}\sum_{i=1}^{N}\left(x_{i,d}-\mu_d\right)^2}$$

A value is normalized with these statistics, and a normalized value is converted back with the inverse:

$$z_{i,d}=\frac{x_{i,d}-\mu_d}{\sigma_d}, \qquad x_{i,d}=z_{i,d}\,\sigma_d+\mu_d$$

Where:
* $N$ - number of real frames in all training episodes together.
* $i$ - index of one frame, $1 \le i \le N$.
* $d$ - index of one feature: one of the 8 state values or one of the 7 action values. States and actions get separate statistics.
* $x_{i,d}$ - raw value of feature $d$ at frame $i$, in that feature's units.
* $\mu_d$, $\sigma_d$ - mean and standard deviation of feature $d$, in the same units.
* $z_{i,d}$ - normalized value; it has no unit.

Use `ddof=0` for the population standard deviation. If a feature is constant, $\sigma_d=0$; set its scale to 1 to avoid division by zero.

Do not fit on overlapping windows: central actions would enter the estimate more often than end actions. Do not fit on padding or validation rows. Use the same saved training statistics for training, validation, and later inference.

**Worked example:** training values $[1,3]$ give $\mu=2$, $\sigma=1$, and normalized values $[-1,1]$. A validation value of 100 becomes 98. That large value is useful evidence of a distribution difference; adding validation to the fit would hide it.

## Exercise 3.1: Fit the feature statistics

Inputs:
* `episodes` - `Mapping[int, TrainingEpisode]`: episode ID to the complete episode (see above).
* `train_ids` - the supplied, nonempty, unique list of training episode IDs.

Return a `FeatureStats`, defined in the import cell:

```python
FeatureStats(
    states=MeanStd(mean=..., std=...),   # each vector has shape (8,)
    actions=MeanStd(mean=..., std=...),  # each vector has shape (7,)
)
```

Concatenate only the specified episodes, compute each feature separately, and protect constant features. Invalid or nonfinite input must produce a useful error.

<details><summary>Hint</summary>Read the full <code>episode.states</code> and <code>episode.actions</code> matrices. Concatenate on the frame axis. Reduce on axis 0. Do not read <code>episode.images</code> when fitting statistics.</details>

In [ ]:
def small_episode(values: Sequence[float], image_level: float) -> TrainingEpisode:
    """Supply a complete numeric episode and a decoded image pair at every frame."""
    values_array = np.asarray(values, dtype=np.float32)
    image = np.full((3, 8, 8), image_level, dtype=np.float32)
    return TrainingEpisode(
        states=np.repeat(values_array[:, None], 8, axis=1),
        actions=np.repeat(values_array[:, None], 7, axis=1),
        images={frame: CameraFrame(image=image, image2=image) for frame in range(len(values))},
    )


EPISODES = {0: small_episode([1, 3], 0.25), 1: small_episode([100, 200], 0.75)}
TRAIN_IDS, VALIDATION_IDS = [0], [1]

In [ ]:
def fit_feature_stats(
    episodes: Mapping[int, TrainingEpisode], train_ids: Sequence[int]
) -> FeatureStats:
    """Fit population statistics on each real training frame once."""
    if not train_ids or len(set(train_ids)) != len(train_ids):
        raise ValueError("train_ids must be nonempty and unique")

    def fit(matrices: list[np.ndarray]) -> MeanStd:
        rows = np.concatenate([np.asarray(matrix, dtype=np.float64) for matrix in matrices])
        if rows.ndim != 2 or not len(rows) or not np.isfinite(rows).all():
            raise ValueError("training features must be finite, nonempty matrices")
        std = rows.std(axis=0, ddof=0)
        return MeanStd(mean=rows.mean(axis=0), std=np.where(std == 0, 1.0, std))

    return FeatureStats(
        states=fit([episodes[i].states for i in train_ids]),
        actions=fit([episodes[i].actions for i in train_ids]),
    )

In [ ]:
STATS = fit_feature_stats(EPISODES, TRAIN_IDS)
np.testing.assert_allclose(STATS.actions.mean, 2)
np.testing.assert_allclose(STATS.actions.std, 1)
changed_validation = {**EPISODES, 1: small_episode([9999, -9999], 0.75)}
CHECK_STATS = fit_feature_stats(changed_validation, TRAIN_IDS)
np.testing.assert_array_equal(CHECK_STATS.actions.mean, STATS.actions.mean)
constant = fit_feature_stats({0: small_episode([4, 4], 0.1)}, [0])
np.testing.assert_array_equal(constant.states.std, np.ones(8))
print("Statistics use training frames only; constant-feature scales are safe.")

## Windows and padding define the target

Let an episode have $L$ frames and let $s$ be the start frame. Target slot $j$ holds the normalized action at frame $s+j$ if that frame exists, and zero otherwise. The padding mask records which slots are absent:

$$y_j=\begin{cases} z_{s+j} & \text{if } s+j<L \\ 0 & \text{otherwise}\end{cases}, \qquad m_j=\begin{cases} \text{False} & \text{if } s+j<L \\ \text{True} & \text{otherwise}\end{cases}$$

Where:
* $L$ - number of frames in the episode.
* $s$ - start frame of the example, $0 \le s < L$.
* $T$ - horizon: the number of target slots.
* $j$ - index of a target slot, $0 \le j < T$.
* $z_{s+j}$ - the 7 normalized action values at frame $s+j$.
* $y_j$ - the target in slot $j$; the stacked targets form `action`.
* $m_j$ - the padding flag of slot $j$; the stacked flags form `action_is_pad`. `True` means the target is absent. It is not a command to execute.

For an episode with actions $[10,20,30]$, start $s=1$, horizon $T=4$, and action statistics $\mu=20$, $\sigma=10$, the target is $[0,1,0,0]$ and the mask is $[\text{False},\text{False},\text{True},\text{True}]$. The zero in slot 0 is a real normalized action; the other zeros are padding. The mask distinguishes them. The training loss will exclude padded slots in the next lesson.

A batch axis combines independent examples. With $B$ examples and images of $H \times W$ pixels, return a `dict[str, torch.Tensor]`. Its keys are `FeatureKey` members from `vla_lab.teaching.records`. A `FeatureKey` is also a `str` (for example, `FeatureKey.STATE == "observation.state"`), so LeRobot can read the dictionary directly:

| Key | Shape and type | Meaning |
|---|---|---|
| `FeatureKey.IMAGE`, `FeatureKey.IMAGE2` | $(B,3,H,W)$ float32 | agent-view and wrist images, already scaled |
| `FeatureKey.STATE` | $(B,8)$ float32 | current state, standardized |
| `FeatureKey.ACTION` | $(B,T,7)$ float32 | standardized future action window |
| `FeatureKey.ACTION_IS_PAD` | $(B,T)$ bool | True only for absent action slots |

`FeatureKey.IMAGE` holds `CameraFrame.image` and `FeatureKey.IMAGE2` holds `CameraFrame.image2`. The same module also exports `CAMERA_KEYS`, the pair `(FeatureKey.IMAGE, FeatureKey.IMAGE2)`.

`horizon` is $T$: a positive number of target slots. It is the predicted chunk length, not the number of actions executed before feedback. `starts` is a nonempty sequence of `(episode_id, start_frame)` pairs supplied by the sampler. A start must be inside its episode and have decoded images, i.e. `start in episodes[episode_id].images`. A larger $B$ increases memory use; a larger $T$ adds target slots and more padding near the end.

## Exercise 3.2: Assemble a complete batch

Implement the complete conversion from selected episodes to model tensors. Read the camera pair at each start, normalize its current state, build a window that stays in that episode, and pad the missing normalized targets. Stack the examples on a new batch axis. `torch.from_numpy` turns an array into a tensor; no gradients are needed for these data tensors.

Use the two operations from this notebook in the local-data experiment below. Do not call a second saved preprocessor on this batch: it is already normalized.

<details><summary>Hint</summary>For each start, <code>count = min(horizon, L - start)</code>. Allocate a zero target and an all-True mask; fill the first <code>count</code> rows and clear those mask entries. Normalize only those real action rows.</details>

In [ ]:
def build_training_batch(
    episodes: Mapping[int, TrainingEpisode],
    starts: Sequence[tuple[int, int]],
    stats: FeatureStats,
    *,
    horizon: int,
) -> dict[str, torch.Tensor]:
    """Combine cameras, normalized state, and episode-bounded action targets."""
    if not starts or horizon < 1:
        raise ValueError("starts must be nonempty and horizon must be positive")
    examples = []
    for episode_id, start in starts:
        episode = episodes[episode_id]
        states = np.asarray(episode.states, dtype=np.float32)
        actions = np.asarray(episode.actions, dtype=np.float32)
        if states.shape != (len(actions), 8) or actions.shape[1:] != (7,):
            raise ValueError("one episode needs aligned state (L,8) and action (L,7)")
        if not 0 <= start < len(actions):
            raise ValueError("start must select a frame inside this episode")
        cameras = episode.images.get(start)
        if cameras is None:
            raise ValueError("no decoded camera pair at the selected start")
        item: dict[str, np.ndarray] = {}
        for key, raw_image in zip(CAMERA_KEYS, (cameras.image, cameras.image2), strict=True):
            image = np.asarray(raw_image, dtype=np.float32)
            if image.ndim != 3 or image.shape[0] != 3 or not np.isfinite(image).all():
                raise ValueError("decoded images must be finite CHW arrays")
            if image.min() < 0 or image.max() > 1:
                raise ValueError("decoded images must already use [0,1] scaling")
            item[key] = image
        item[FeatureKey.STATE] = ((states[start] - stats.states.mean) / stats.states.std).astype(
            np.float32
        )
        count = min(horizon, len(actions) - start)
        target = np.zeros((horizon, 7), dtype=np.float32)
        target[:count] = (actions[start : start + count] - stats.actions.mean) / stats.actions.std
        mask = np.ones(horizon, dtype=np.bool_)
        mask[:count] = False
        item[FeatureKey.ACTION] = target
        item[FeatureKey.ACTION_IS_PAD] = mask
        examples.append(item)
    return {key: torch.from_numpy(np.stack([row[key] for row in examples])) for key in examples[0]}

In [ ]:
BATCH = build_training_batch(EPISODES, [(0, 1), (1, 0)], STATS, horizon=4)
assert BATCH[FeatureKey.ACTION].shape == (2, 4, 7)
assert BATCH[FeatureKey.STATE].shape == (2, 8)
torch.testing.assert_close(BATCH[FeatureKey.STATE][:, 0], torch.tensor([1.0, 98.0]))
assert BATCH[FeatureKey.ACTION_IS_PAD].dtype == torch.bool
assert BATCH[FeatureKey.ACTION_IS_PAD].tolist() == [
    [False, True, True, True],
    [False, False, True, True],
]
torch.testing.assert_close(BATCH[FeatureKey.ACTION][0, :, 0], torch.tensor([1.0, 0.0, 0.0, 0.0]))
assert BATCH[FeatureKey.ACTION][1, 0, 0] == 98
assert BATCH[FeatureKey.IMAGE][0].mean() == 0.25
np.testing.assert_allclose(
    BATCH[FeatureKey.ACTION][0, 0].numpy() * STATS.actions.std + STATS.actions.mean, np.full(7, 3)
)
try:
    build_training_batch(EPISODES, [(0, 2)], STATS, horizon=4)
except ValueError as error:
    print("Expected invalid-start error:", error)
else:
    raise AssertionError("A start outside its episode must fail")
print({str(key): (tuple(value.shape), str(value.dtype)) for key, value in BATCH.items()})

## Visual experiment: change the scale

The plot shows raw values and their normalized values. The slider changes only this small calculation. It does not change saved statistics or run training. Compare a small scale with a large scale: which gives a larger normalized error for the same input difference?


In [ ]:
def show_scale(std: float = 1.0) -> None:
    raw = np.array([1.0, 2.0, 3.0])
    fig, ax = plt.subplots(figsize=(6, 2.5))
    ax.plot(raw, (raw - 2) / std, "o-")
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.set(xlabel="Raw feature value", ylabel="Normalized value", title=f"Mean 2; scale {std:g}")
    plt.show()


show_scale()
display(interactive(show_scale, std=FloatSlider(value=1, min=0.25, max=4, step=0.25)))

## Required local-data experiment: use your batch in ACT

The supplied reader checks the frozen split and decodes camera pairs. Read two training episodes and one validation episode of `bowl_plate`, with three decoded image frames per episode. Fit this experiment's statistics on those two training episodes only. These small-subset statistics are for this batch exercise; later training lessons use the full declared training split and save its processors.

`image_samples_per_episode=3` limits video decoding to three evenly spaced frames (the first, a middle, and the last frame), so each `episode.images` has three keys. It does not limit the action and state rows used for windows or statistics. Each example below starts at the last decoded frame, `max(episode.images)`. `horizon=20` gives twenty target slots. The next cell builds a two-example batch with your function and passes it into a supplied ACT model. Random model weights check the data interface, not task quality. All assets must be local; missing files produce a setup error.

In [ ]:
SPLIT = yaml.safe_load((WORKSPACE / "configs/data/episodes.yaml").read_text())["tasks"][
    "bowl_plate"
]
LOCAL_TRAIN_IDS = SPLIT["train_episode_ids"][:2]
LOCAL_VALID_IDS = SPLIT["validation_episode_ids"][:1]
assert not set(LOCAL_TRAIN_IDS) & set(LOCAL_VALID_IDS)
LOCAL_EPISODES = read_local_episode_rows(
    "bowl_plate",
    LOCAL_TRAIN_IDS + LOCAL_VALID_IDS,
    workspace=WORKSPACE,
    image_samples_per_episode=3,
)
LOCAL_STATS = fit_feature_stats(LOCAL_EPISODES, LOCAL_TRAIN_IDS)
# The last decoded frame of each episode: its window runs past the episode end.
LOCAL_STARTS = [
    (i, max(LOCAL_EPISODES[i].images)) for i in (LOCAL_TRAIN_IDS[0], LOCAL_VALID_IDS[0])
]
LOCAL_BATCH = build_training_batch(LOCAL_EPISODES, LOCAL_STARTS, LOCAL_STATS, horizon=20)
print("Episode lengths:", {i: e.frame_count for i, e in LOCAL_EPISODES.items()})
print("Padding counts:", LOCAL_BATCH[FeatureKey.ACTION_IS_PAD].sum(dim=1).tolist())

In [ ]:
torch.manual_seed(17)
POLICY = ACTPolicy(make_act_config(device="cpu", chunk_size=20))
RAW_MODEL = make_act_raw_model(POLICY).train()
with torch.no_grad():
    PREDICTED, (MU, LOGVAR) = RAW_MODEL(LOCAL_BATCH)
assert PREDICTED.shape == LOCAL_BATCH[FeatureKey.ACTION].shape
assert torch.isfinite(PREDICTED).all() and MU is not None and LOGVAR is not None
print("Your batch reached ACT:", tuple(PREDICTED.shape))

## Interpret the result

Compare the last-frame examples with an example from the middle of an episode. Explain the mask counts and why a zero target cannot identify padding on its own. Change the validation values in the CPU fixture and confirm that training statistics stay unchanged. Then explain why a checkpoint must carry the statistics used in training.

**Optional extension:** compare statistics from two training episodes with the full frozen training statistics in `configs/data/stats/bowl_plate.json`. Report the selected episode IDs. Do not use the validation split to choose new statistics.
